In [13]:
import pandas as pd
import datetime as dt

In [103]:
bch_data = pd.read_excel("https://www.bch.hn/estadisticos/GIE/LIBTipo%20de%20cambio/Precio%20Promedio%20Diario%20del%20D%C3%B3lar.xlsx",header=6)[["Fecha","Compra 1/"]].rename(columns={"Compra 1/":"Tasa"})
hnl_usd = bch_data[bch_data["Fecha"].map(lambda x: isinstance(x,dt.datetime))]
hnl_usd["Fecha"] = pd.to_datetime(hnl_usd["Fecha"], format="%Y-%m-%d 00:00:00")

In [83]:
import xmltodict, requests
import numpy as np

In [101]:
eib_data = xmltodict.parse(requests.get("https://www.ecb.europa.eu/stats/policy_and_exchange_rates/euro_reference_exchange_rates/html/usd.xml").content)

In [99]:
eib_data = xmltodict.parse(requests.get("https://www.ecb.europa.eu/stats/policy_and_exchange_rates/euro_reference_exchange_rates/html/usd.xml").content)
usd_eur = pd.DataFrame(eib_data['CompactData']['DataSet']['Series']['Obs']).rename(columns={'@TIME_PERIOD':'Fecha','@OBS_VALUE':'Tasa'})[['Fecha','Tasa']]
usd_eur["Fecha"] = pd.to_datetime(eur_usd["Fecha"],format='%Y-%m-%d')

In [56]:
import psycopg2

In [57]:
dbname = 'trans_app'
host = 'localhost'
port = 5432
cnxn = psycopg2.connect(dbname=dbname, host=host, port=port)

In [71]:
query = 'SELECT * from (SELECT Cuenta, sum(Monto) as Balance, Moneda FROM transacciones t join cuentas c on t.Cuenta = c."Nombre Cuenta" Group by Cuenta, Moneda) Where (Balance <-0.01 or Balance >0.01)'

In [72]:
bal = pd.read_sql(query,cnxn)

/tmp/ipykernel_24042/2990908830.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  bal = pd.read_sql(query,cnxn)


In [73]:
bal.head()

,cuenta,balance,moneda
0,Millas Plus ($),-165.18,USD
1,Radiodiagnóstico,495297.32,HNL
2,Ficohsa,67400.91,HNL
3,RAP,505900.46,HNL
4,Cuenta Dólares BAC $,443.52,USD


In [123]:
tasa_usd = usd_eur[usd_eur["Fecha"]-dt.datetime(2021,12,5) <= dt.timedelta(0)].tail(1)["Tasa"]
tasa_usd

5869    1.1291
Name: Tasa, dtype: object

In [130]:
def to_eur(data:pd.DataFrame, date:dt.datetime) -> float:
    tasa_usd = float(usd_eur[usd_eur["Fecha"]-date <= dt.timedelta(0)].tail(1)["Tasa"])
    tasa_hnl = float(hnl_usd[hnl_usd["Fecha"]-date <= dt.timedelta(0)].tail(1)["Tasa"])*tasa_usd
    return data.apply(lambda x: x[1]/(1 if x[2]=="EUR" else (tasa_usd if x[2]=="USD" else tasa_hnl)), axis=1)

def to_hnl(data:pd.DataFrame, date:dt.datetime) -> float:
    tasa_usd = float(hnl_usd[hnl_usd["Fecha"]-date <= dt.timedelta(0)].tail(1)["Tasa"])
    tasa_eur = float(usd_eur[usd_eur["Fecha"]-date <= dt.timedelta(0)].tail(1)["Tasa"])*tasa_usd
    return data.apply(lambda x: x[1]*(1 if x[2]=="HNL" else (tasa_usd if x[2]=="USD" else tasa_eur)), axis=1)

def to_usd(data:pd.DataFrame, date:dt.datetime) -> float:
    tasa_eur = 1./float(usd_eur[usd_eur["Fecha"]-date <= dt.timedelta(0)].tail(1)["Tasa"])
    tasa_hnl = float(hnl_usd[hnl_usd["Fecha"]-date <= dt.timedelta(0)].tail(1)["Tasa"])
    return data.apply(lambda x: x[1]/(1 if x[2]=="USD" else (tasa_eur if x[2]=="EUR" else tasa_hnl)), axis=1)

In [134]:
bal["balance eur"] = to_usd(bal, dt.datetime(2021,12,5))
bal

,cuenta,balance,moneda,balance eur
0,Millas Plus ($),-165.18,USD,-165.180000
1,Radiodiagnóstico,495297.32,HNL,20511.583952
2,Ficohsa,67400.91,HNL,2791.251574
3,RAP,505900.46,HNL,20950.688279
4,Cuenta Dólares BAC $,443.52,USD,443.520000
5,Activos Fijos,494000.00,HNL,20457.858468
6,Billetera EUR,185.00,EUR,208.883500
7,Infinite,140845.00,HNL,5832.767360
8,Ficopen,783410.03,HNL,32443.100235
9,KBC,11927.27,EUR,13467.080557


In [140]:
bal.to_dict(orient='records')

[{'cuenta': 'Millas Plus ($)',
  'balance': -165.17999999998835,
  'moneda': 'USD',
  'balance eur': -165.17999999998835},
 {'cuenta': 'Radiodiagnóstico',
  'balance': 495297.32000000007,
  'moneda': 'HNL',
  'balance eur': 20511.583951762525},
 {'cuenta': 'Ficohsa',
  'balance': 67400.90999999737,
  'moneda': 'HNL',
  'balance eur': 2791.2515736813116},
 {'cuenta': 'RAP',
  'balance': 505900.46,
  'moneda': 'HNL',
  'balance eur': 20950.688278558177},
 {'cuenta': 'Cuenta Dólares BAC $',
  'balance': 443.51999999999873,
  'moneda': 'USD',
  'balance eur': 443.51999999999873},
 {'cuenta': 'Activos Fijos',
  'balance': 494000.0,
  'moneda': 'HNL',
  'balance eur': 20457.85846806255},
 {'cuenta': 'Billetera EUR',
  'balance': 184.99999999999997,
  'moneda': 'EUR',
  'balance eur': 208.88349999999994},
 {'cuenta': 'Infinite',
  'balance': 140844.99999999965,
  'moneda': 'HNL',
  'balance eur': 5832.7673601908145},
 {'cuenta': 'Ficopen',
  'balance': 783410.03,
  'moneda': 'HNL',
  'balance